In [4]:
import pandas as pd
import re
df = pd.read_csv('jd_data_raw.csv', encoding='utf-8')

In [5]:
# 清洗全文

def clean_content(text):
    if not isinstance(text, str):
        return text
    
    # 删除 "NEW " 字段
    text = text.replace('NEW ', '')
    
    # 任务2：删除 "求人票No ... あとからチェックしたい方は 気になる" 之后的所有内容
    # 我们定位到 "気になる" 这个关键词并截断
    # 如果关键词非常固定，可以使用 split
    keyword = "あとからチェックしたい方は 気になる"
    if keyword in text:
        text = text.split(keyword)[0]
    
    # 额外清理：由于截断后末尾可能残留 "求人票No K..." 等信息，
    # 我们可以用正则表达式进一步精准定位到 "求人票No" 之前
    text = re.sub(r'求人票No\s+K\d{8}-\d{3}-\d{2}-\d{2}.*$', '', text, flags=re.DOTALL)
    
    return text.strip()

df['内容全文'] = df['内容全文'].apply(clean_content)




In [6]:
# 提取position和会社

def extract_advanced(text):
    if not isinstance(text, str):
        return None, None

    # 1. 定义公司名的常见特征（匹配包含 株式会社/合同会社/有限会社/.. 的连续词段）
    # \S* 表示匹配非空白字符，确保把公司名完整抓取
    company_pattern = r"(\S*(?:株式会社|合同会社|有限会社|（株）|\(株\))\S*)"
    
    # 2. 定义职位和公司之后的“护城河”标签（以此为终点防止匹配过度）
    stop_tags = r"(?:業界未経験|フレックス|年間休日|土曜出勤なし|正社員|想定年収|勤務地)"
    
    # 尝试匹配：职位名 + 空间 + 公司名 + (空间 + 标签)
    # ^(.*?)\s+ ：从开头非贪婪匹配，直到遇到空格，这是职位名
    # {company_pattern} ：匹配包含公司特征的部分
    # (?:\s+{stop_tags}|$) ：断言后面跟着的是标签或者是字符串结尾
    regex = rf"^(.*?)\s+{company_pattern}(?:\s+{stop_tags}|$)"
    
    match = re.search(regex, text)
    
    if match:
        job_title = match.group(1).strip()
        company_name = match.group(2).strip()
        
        # 针对 NEC 这种带括号的特殊处理：日本電気株式会社（ＮＥＣ）
        # 如果公司名后面紧跟一个括号，尝试把它也抓进来
        extra_parentheses = re.search(r"^[（\(][^）\)]+[）\)]", text[match.end(2):])
        if extra_parentheses:
            company_name += extra_parentheses.group()
            
        return job_title, company_name
    
    # 保底方案：如果没找到“株式会社”等关键词，按最后一个空格前的逻辑切分
    parts = text.split()
    if len(parts) >= 2:
        # 寻找列表里包含“单位”特征的索引
        for i, p in enumerate(parts):
            if any(x in p for x in ['株式会社', '合同会社', '有限会社', '（株）']):
                return " ".join(parts[:i]), parts[i]
                
    return text[:20], "未识别" # 极端情况返回前20字

# 应用
df[['Position', '会社名']] = df['内容全文'].apply(lambda x: pd.Series(extract_advanced(x)))


In [8]:


# 提取其他信息
def extract_fields(text):
    # 这里的 text 已经是之前处理过的（删除了 NEW 和 尾部冗余）
    clean_text = text

    
    if not isinstance(clean_text, str) or clean_text.strip() == "":
        return pd.Series([''] * 9)

    # === 3. 提取在宅/Remote 关键词 (优化版) ===
    # 加入了 ハイブリッド、出社 等关键词，并捕捉前后 25 个字符以获取更完整的语境（如：原则出社、不可等）
    # 使用 re.IGNORECASE 以防万一有英文大写
    pattern = r'(.{0,2}(?:在宅|リモート|テレワーク|ハイブリッド|出社).{0,25})'
    remote_keywords = re.findall(pattern, clean_text)
    
    if remote_keywords:
        # 去重并合并
        remote_info = " || ".join(dict.fromkeys([k.strip() for k in remote_keywords]))
    else:
        remote_info = "无匹配"

    # === 4. 提取特定段落 ===
    def extract_between(source, start_str, end_str):
        start_idx = source.find(start_str)
        if start_idx == -1:
            return ""
        start_idx += len(start_str)
        end_idx = source.find(end_str, start_idx)
        if end_idx == -1:
            # 如果没找到结束标志，截取后面 500 字
            return source[start_idx:].strip()[:500] 
        return source[start_idx:end_idx].strip()

    job_content = extract_between(clean_text, "仕事の内容", "配属先情報")
    dept_info = extract_between(clean_text, "配属先情報", "必要な能力・経験")
    requirements = extract_between(clean_text, "必要な能力・経験", "勤務地") 
    salary = extract_between(clean_text, "給与", "就業時間")
    work_hours = extract_between(clean_text, "就業時間", "通勤手当")
    selection = extract_between(clean_text, "選考内容", "企業情報")
    
    # 提取员工人数
    emp_count_match = re.search(r'従業員数[\s　]*([0-9,]+)', clean_text)
    emp_count = emp_count_match.group(1) if emp_count_match else ""

    return pd.Series([
        emp_count, 
        remote_info, 
        salary,
        requirements, 
        job_content,  
        dept_info, 
        work_hours, 
        selection, 
        clean_text
    ])

# 应用函数到 DataFrame
columns = [
    '従業員数',
    '在宅可否', 
    '給与',
    '必要経験', 
    '仕事内容',  
    '配属部门', 
    '就業時間', 
    '選考内容', 
    '清洗后全文'
]

df[columns] = df['内容全文'].apply(extract_fields)

# 只有在确定不再需要原始列时才 drop
if '内容全文' in df.columns:
    df = df.drop(columns = ['内容全文'])

# 保存
df.to_csv('jd_data_structured.csv', index=False, encoding='utf-8-sig')

In [9]:
# ===== 2. role 字典 =====
ROLE_DICT = {
    "DA": [
        "データアナリスト",
        "データ分析",
        "データ集計",
        "マーケティングデータ"
    ],
    "DS": [
        "データサイエンティスト",
        "数理",
        "AI",
        "機械学習"
    ],
    "DE": [
        "データエンジニア",
        "基盤",
        "構築"
        "DWH",
        "ETL",
        "Databricks"
    ],
    "MLE": [
        "機械学習エンジニア",
        "MLエンジニア",
        "AIエンジニア"
    ],
    "BI": [
        "BI",
        "ダッシュボード",
        "可視化"
    ],
    "CONSULTANT": [
        "コンサル",
        "利活用",
        "データ活用"
    ]
}

# ===== 3. 优先级（很关键）=====
PRIORITY = ["DS", "MLE", "DE", "DA", "BI", "CONSULTANT"]


# ===== 4. 分类函数 =====
def classify_role(title):
    matched_roles = []

    for role, keywords in ROLE_DICT.items():
        for kw in keywords:
            if kw in title:
                matched_roles.append(role)
                break

    if not matched_roles:
        return "OTHER"

    # 按优先级返回
    for p in PRIORITY:
        if p in matched_roles:
            return p


# ===== 5. 应用 =====
df["role_type"] = df["Position"].apply(classify_role)

print(df[["Position", "role_type"]].head(20))

                                         Position role_type
0            【データ活用/教育枠】佐川急便が抱えるビジネス課題をデータ分析の力で解決        DA
1           【4/1入社限定】データアナリスト/基盤構築エンジニア/自由な働き方可能◎        DE
2           【3/1入社限定】データアナリスト/基盤構築エンジニア/自由な働き方可能◎        DE
3              【AIプロダクトマネージャー】テクノロジー戦略推進本部/グロース上場        DS
4       【2/1～4/1入社限定】データアナリスト/基盤構築エンジニア/自由な働き方可能◎        DE
5        【AI/DX人材育成コンサルタント】顧客のAI/DX人材を育成/データサイエンス        DS
6               【データアナリスト】業界No.1ロボアドバイザー◎MUFGグループ        DA
7       【データ分析】CX改善/バックオフィス経験が活かせる/週3リモート可/所定労働7H        DA
8           【データアナリスト】SQL活用／急成長コミュニティアプリ／ユーザー体験向上        DA
9        【プロジェクトマネージャー】国内No.1会員数＆開催数のAI/データのコンペ運営        DS
10               【プロダクトマネージャー】テクノロジー戦略推進本部/グロース上場     OTHER
11    【未経験/データ活用支援コンサルタント】Tableau/PwerBIカスタマーサクセス        BI
12         【マーケティング戦略企画】◎急成長企業の自社マーケ/平均残業20h/裁量権高     OTHER
13          【動作解析エンジニア】ヨガ・ピラティス事業を展開する成長企業/IPO準備中     OTHER
14                  【営業企画/メンバークラス】KDDIグループ/成長中の企業     OTHER
15                  【営業企画/リーダークラス】KDDIグル

In [12]:


FULL_REMOTE_KW = [
    "フルリモート",
    "完全在宅",
    "フルリモート可"
    "基本在宅",
    "基本リモート"
]

HYBRID_KW = [
    "リモートワーク可",
    "在宅勤務",
    "リモート可",
    "ハイブリッド",
    "在宅OK",
    "リモート推奨",
    "基本在宅",
    "週",   # 週2日など
]

ONSITE_KW = [
    "出社",
    "常駐",
]

# ===== 3. 否定关键词（关键新增）=====
NEGATIVE_FULL_REMOTE = [
    "フルリモート不可",
    "完全在宅不可"
]

NEGATIVE_REMOTE = [
    "リモート不可",
    "在宅不可"
]

# ===== 4. 分类函数 =====

def classify_remote(text):
    if pd.isna(text):
        return "UNKNOWN", -1

    text = str(text)

    # ---- 否定优先（最重要）----
    for kw in NEGATIVE_FULL_REMOTE:
        if kw in text:
            return "ONSITE", 0

    for kw in NEGATIVE_REMOTE:
        if kw in text:
            return "ONSITE", 0

    # ---- FULL REMOTE ----
    for kw in FULL_REMOTE_KW:
        if kw in text:
            return "FULL_REMOTE", 2

    # ---- HYBRID ----
    for kw in HYBRID_KW:
        if kw in text:
            return "HYBRID", 1

    # ---- ONSITE ----
    if any(kw in text for kw in ONSITE_KW):
        return "ONSITE", 0

    return "UNKNOWN", -1


# ===== 5. 应用 =====

df[["remote_type", "remote_score"]] = df["在宅可否"].apply(
    lambda x: pd.Series(classify_remote(x))
)

# ===== 6. 查看结果 =====
print(df[["在宅可否", "remote_type", "remote_score"]].head(20))

# ===== 7. 分布 =====
print("\n=== remote_type 分布 ===")
print(df["remote_type"].value_counts())

                                                 在宅可否 remote_type  \
0   力】リモートワーク基本 ★入社者のスムーズな定着を目的に、入社 || 後は出社をいただき先輩...      HYBRID   
1   考 在宅勤務可※在宅勤務/リモートワークの利用には条件があ || 備 在宅勤務（全従業員利用...      HYBRID   
2   考 在宅勤務可※在宅勤務/リモートワークの利用には条件があ || 備 在宅勤務（全従業員利用...      HYBRID   
3   境】ハイブリッドワーク（基本週3日出社推奨） チームによって出社日 || や出社日数が異なり...      HYBRID   
4   考 在宅勤務可※在宅勤務/リモートワークの利用には条件があ || 備 在宅勤務（全従業員利用...      HYBRID   
5   4日リモートOK/年間休日126日/全社平均残業時間約10時間 || 備 在宅勤務（全従業員...      HYBRID   
6   一部リモート 【ツール】仕様策定：Figmaプロジェクトマネジ || 備 在宅勤務（全従業員...      HYBRID   
7   週3リモート可/所定労働7H 株式会社スカパー・カスタマーリレ || までリモートワーク可能...      HYBRID   
8   （※出社頻度は2日/週） 転勤 無 勤務条件 雇用形態 正 || 備 在宅勤務（全従業員利用...      HYBRID   
9   4日リモートOK/年間休日126日/全社平均残業時間約10時間 || 備 在宅勤務（全従業員...      HYBRID   
10  境】ハイブリッドワーク（基本週3日出社推奨） チームによって出社日 || や出社日数が異なり...      HYBRID   
11                      備 在宅勤務（一部従業員利用可） リモートワーク可（一部従      HYBRID   
12                      には出社スタイルです。  業務内容の変更の範囲：当社業務全      ONSITE   
13                                

In [14]:
df.to_csv('test.csv', index=False, encoding='utf-8-sig')